# 02. Physics-Informed Feature Engineering & Memory Construction

This notebook translates the thermodynamic and statistical constraints identified during the Exploratory Data Analysis (EDA) into a robust, tabular dataset suitable for Machine Learning.

In the previous notebook, we established a critical physical reality: predicting the stator winding temperature ($T_{stator}$) is mathematically ill-posed if restricted to instantaneous state variables ($X_t \rightarrow Y_t$). The thermal inertia of the motor demands that our predictive models have access to the history of heat generation and dissipation.

## Objectives
This feature engineering pipeline is designed to capture the temporal dynamics while strictly enforcing a **leakage-safe** environment:

1. **Mathematical Proxies:** Compute instantaneous physical invariants (e.g., Active Power, Torque-Speed envelope) to assist the models in capturing non-linear electromechanical relationships.
2. **Thermal Memory (Rolling Statistics):** Construct historical features using rolling windows. The window sizes are directly motivated by the empirically derived thermal time constant ($\tau \approx 110\text{s}$) identified in the EDA.
3. **Strict Causality Enforcement:** Ensure absolute protection against "future-to-past" data leakage by applying explicit temporal shifting operations and strict profile-based isolation (`profile_id`).

---

## 1. Environment Setup & Data Integrity
We begin by initializing the environment and loading the raw synchronous measurements. The `profile_id` variable is immediately cast as a categorical identifier to ensure it acts purely as a grouping key, never as a numerical feature.

In [104]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import skew, kurtosis
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

DATA_PATH = Path("../data/raw/measures_v2.csv")
assert DATA_PATH.exists(), f"Missing dataset at: {DATA_PATH.resolve()}"

df_raw = pd.read_csv(DATA_PATH)

# profile_id is an identifier, not a numerical feature
df_raw["profile_id"] = df_raw["profile_id"].astype("category")

print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())

df_raw.head()

Shape: (1330816, 13)
Columns: ['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'profile_id']


,u_q,coolant,stator_winding,u_d,stator_tooth,motor_speed,i_d,i_q,pm,stator_yoke,ambient,torque,profile_id
0,-0.450682,18.805172,19.086670,-0.350055,18.293219,0.002866,0.004419,0.000328,24.554214,18.316547,19.850691,0.187101,17
1,-0.325737,18.818571,19.092390,-0.305803,18.294807,0.000257,0.000606,-0.000785,24.538078,18.314955,19.850672,0.245417,17
2,-0.440864,18.828770,19.089380,-0.372503,18.294094,0.002355,0.001290,0.000386,24.544693,18.326307,19.850657,0.176615,17
3,-0.327026,18.835567,19.083031,-0.316199,18.292542,0.006105,0.000026,0.002046,24.554018,18.330833,19.850647,0.238303,17
4,-0.471150,18.857033,19.082525,-0.332272,18.291428,0.003133,-0.064317,0.037184,24.565397,18.326662,19.850639,0.208197,17


### 1.1 Dataset Schema and Operational Constraints

The dataset is loaded successfully with its ~1.3 million timestamps. Each row represents a synchronous sample across 69 independent operational driving profiles.

**Note on Sensor Availability:**
The raw dataset contains the complete ground-truth instrumentation, including `pm` (Permanent Magnet), `stator_tooth`, and `stator_yoke` temperatures. As established in the EDA, these internal variables violate the industrial "Soft-Sensor" constraint, as they are unobservable in standard production environments. While we load them here for structural completeness and potential Oracle baseline tracking, they will be strictly excluded from the final production-ready feature sets at the end of this pipeline.

---

## 2. Signal Taxonomy & Industrial Boundary Conditions

To mathematically formulate the Soft-Sensor problem, we must rigorously define the observational scope of our system. We segregate the raw telemetry into distinct logical arrays. This strict taxonomy guarantees the structural integrity of our pipeline by preventing the inadvertent leakage of unobservable latent states into the production feature space.

This segregation allows us to dual-track our modeling:
1. Constructing a **Production-Realistic Soft-Sensor** constrained by real-world observability.
2. Training an **Oracle Baseline** (a sensor-rich reference model) to quantify the absolute upper bound of predictability.

In [105]:
TARGET = "stator_winding"
GROUP = "profile_id"

BASE_SIGNALS = [
    "u_d","u_q",
    "i_d","i_q",
    "motor_speed","torque",
    "coolant","ambient"
]

THERMAL_INTERNAL = ["pm","stator_tooth","stator_yoke"]

### 2.1 Physical Interpretation of the Feature Sets

The distinction between these sets perfectly mirrors the thermodynamic transfer function of the motor:

**1. The Observable Inputs (`BASE_SIGNALS`):**
These variables constitute the strict inputs of our Soft-Sensor. They represent the active electromechanical states and the thermodynamic boundary conditions:
* **Heat Sources (The Excitations):** Currents ($i_d$, $i_q$) and voltages ($u_d$, $u_q$) dictate the active power injection and subsequent Joule/Copper losses. `motor_speed` and `torque` dictate the mechanical load and subsequent Iron losses (eddy currents).
* **Boundary Conditions (The Heat Sinks):** `coolant` and `ambient` define the reference temperatures driving convective and conductive heat dissipation.

**2. The Unobservable Latent States (`THERMAL_INTERNAL`):**
These variables (`pm`, `stator_tooth`, `stator_yoke`) act as massive internal thermal buffers (capacitors). 
* **Industrial Constraint:** While spatial thermal diffusion gives them near-perfect correlation with the winding temperature, they are physically uninstrumented in a standard electric vehicle or industrial drive. They are strictly quarantined from the main modeling pipeline to preserve the integrity of the industrial scenario.

---

## 3. Pipeline Integrity & Schema Contract (MLOps)

In an industrial deployment environment, silent failures due to upstream data schema drift (e.g., a renamed sensor or a dropped telemetry stream) are a primary cause of model degradation. 

Before proceeding with any mathematical transformations, we enforce a strict *Schema Contract*. This defensive programming step halts execution immediately if the expected signal taxonomy is violated, ensuring the downstream physical equations receive the correct vector inputs.

In [106]:
EXPECTED_COLUMNS = set(BASE_SIGNALS + THERMAL_INTERNAL + [TARGET, GROUP])

missing_cols = EXPECTED_COLUMNS - set(df_raw.columns)
extra_cols = set(df_raw.columns) - EXPECTED_COLUMNS

print("Expected columns:", sorted(EXPECTED_COLUMNS))
print("Missing columns:", sorted(missing_cols))
print("Extra columns:", sorted(extra_cols))

assert len(missing_cols) == 0, f"Missing required columns: {missing_cols}"

Expected columns: ['ambient', 'coolant', 'i_d', 'i_q', 'motor_speed', 'pm', 'profile_id', 'stator_tooth', 'stator_winding', 'stator_yoke', 'torque', 'u_d', 'u_q']
Missing columns: []
Extra columns: []


---

## 4. Physics-Informed Feature Engineering: Instantaneous Thermodynamics

Raw telemetry in the $d-q$ (Park) reference frame is optimized for field-oriented control (FOC) algorithms, not necessarily for thermodynamic modeling. To assist the Machine Learning algorithms in capturing the non-linear heat generation, we map these raw vectors into domain-specific, physically meaningful scalar quantities. 

These new features act as direct proxies for the thermodynamic forcing functions (the heat sources) of the system.

In [107]:
df = df_raw.copy()

df["u_norm"] = np.sqrt(df["u_d"]**2 + df["u_q"]**2)
df["i_norm"] = np.sqrt(df["i_d"]**2 + df["i_q"]**2)

df["p_elec_proxy"] = df["u_d"]*df["i_d"] + df["u_q"]*df["i_q"]
df["omega_torque"] = df["motor_speed"] * df["torque"]

### 4.1 Thermodynamic Interpretation of the Engineered Proxies

We have constructed four instantaneous proxies that represent the momentary energy injection and loss mechanics of the motor:

1. **Current Magnitude ($i_{norm}$):** Mathematically defined as $\sqrt{i_d^2 + i_q^2}$. This is the most critical driver of heating, as the stator's Copper Losses (Joule effect) are strictly proportional to the square of this magnitude ($P_{cu} \propto R_s \cdot i_{norm}^2$).
2. **Voltage Magnitude ($u_{norm}$):** Defined as $\sqrt{u_d^2 + u_q^2}$. It serves as a proxy for the internal magnetic flux state and core saturation, which are linked to Iron Losses (eddy currents and hysteresis).
3. **Active Electrical Power Proxy ($P_{elec}$):** Computed as $u_d i_d + u_q i_q$. This represents the total raw electrical energy entering the stator.
4. **Mechanical Power Proxy ($P_{mech}$):** Computed as $\omega \cdot \tau$. This represents the mechanical energy leaving the rotor.

> **The Energy Balance Insight:**
> By providing the algorithm with both the electrical input proxy and the mechanical output proxy, we implicitly grant it the ability to learn the **Motor Power Loss Equation**. According to the First Law of Thermodynamics, the difference between these two proxies represents the total energy dissipated as heat within the motor housing ($P_{loss} \approx P_{elec} - P_{mech}$). 
> While these features represent *instantaneous* energy rates, they will serve as the foundation for the temporal integration steps in the next section.

---



## 5. Constructing the Thermal Memory: Temporal Aggregation

As established in the EDA, the stator temperature does not react instantly to electrical excitations. Heat transfer is governed by differential equations where the current temperature is the integral of past heat generation and dissipation rates. Tabular Machine Learning models (like Gradient Boosting) cannot natively compute these integrals. 

To overcome this, we explicitly construct the **Thermal Memory** of the system by generating time-aware rolling statistics (moving averages and standard deviations) over multi-scale historical horizons.

### 5.1 Strictly Causal Windowing Paradigm

> **⚠️ Critical MLOps Constraint: Zero Temporal Peeking**
> To prevent future-to-past data leakage, all rolling features are computed in a strictly causal manner. By applying a `.shift(1)` operation before the `.rolling()` window, we guarantee that the prediction at time $t$ relies *only* on data from $t-W$ up to $t-1$. Furthermore, all aggregations are strictly isolated within their respective `profile_id` via group-by operations, preventing inter-session contamination.


### 5.2 Physically Motivated Time Scales

The rolling window sizes ($W$) are not arbitrary hyperparameters; they are directly derived from the cross-correlation analysis in our EDA, mapped to the $2\text{ Hz}$ sampling frequency:
* **Short-Term Dynamics (5s $\rightarrow$ 10 samples):** Captures high-frequency electrical transients and immediate torque adjustments.
* **Mid-Term Memory (25s $\rightarrow$ 50 samples):** Captures intermediate thermal buffering, such as the initial heat transfer from the copper windings to the surrounding air gap.
* **Long-Term Thermal Inertia (110s $\rightarrow$ 220 samples):** Matches the empirically estimated dominant thermal time constant ($\tau$), capturing deep thermal soaking within the motor core.

In [109]:
FS_HZ = 2.0  # sampling frequency

ROLL_WINDOWS = [
    int(5 * FS_HZ),     # ~5 s  (short-term dynamics)
    int(25 * FS_HZ),    # ~25 s (mid-term memory)
    int(110 * FS_HZ),   # ~110 s (thermal inertia from EDA)
]

g = df.groupby(GROUP, group_keys=False)

for w in ROLL_WINDOWS:
    for c in BASE_SIGNALS + THERMAL_INTERNAL:
        df[f"{c}_rm{w}"] = g[c].transform(lambda s: s.shift(1).rolling(w, min_periods=w).mean())
        df[f"{c}_rs{w}"] = g[c].transform(lambda s: s.shift(1).rolling(w, min_periods=w).std())

---

## 6. Cumulative Excitation: From Power to Energy

While rolling averages of raw signals (like speed or voltage) provide context, the true driver of thermal elevation is **Accumulated Energy** (measured in Joules).

According to the First Law of Thermodynamics, energy is the integral of power over time ($E = \int P dt$). By computing the rolling average of our instantaneous Active Power Proxy ($P_{elec\_proxy}$) and our Joule heating proxy ($i_{norm}$) over the same time windows, we mathematically approximate this integral. 

These resulting features (`p_elec_proxy_ra{W}` and `i_norm_ra{W}`) act as direct **Cumulative Thermal Excitation Proxies**, providing the model with a quantifiable measure of the total heat injected into the stator over the last $W$ seconds.

In [ ]:
for w in ROLL_WINDOWS:
    df[f"p_elec_proxy_ra{w}"] = g["p_elec_proxy"].transform(
    lambda s: s.shift(1).rolling(w, min_periods=w).mean()
)
    df[f"i_norm_ra{w}"] = g["i_norm"].transform(
    lambda s: s.shift(1).rolling(w, min_periods=w).mean()
)

## 7. Transient Dynamics: Discrete Temporal Gradients

While rolling averages capture long-term energy accumulation, they inherently act as low-pass filters, smoothing out rapid operational changes. To equip the model with the ability to detect sudden load variations (e.g., hard acceleration or emergency braking), we must capture the high-frequency dynamics.

We achieve this by computing first-order differences, which serve as a discrete numerical approximation of the continuous time derivative $\frac{dx}{dt}$:
* `diff(1)` ($\approx 0.5\text{s}$ window): Captures instantaneous shock dynamics (e.g., inductive spikes where $V = L \frac{di}{dt}$ or angular acceleration $\alpha = \frac{d\omega}{dt}$).
* `diff(5)` ($\approx 2.5\text{s}$ window): Captures slightly smoother short-term trends, mitigating sensor noise while highlighting regime transitions.

In [96]:
for c in BASE_SIGNALS:
    df[f"{c}_diff1"] = g[c].transform(lambda s: s.diff(1))
    df[f"{c}_diff5"] = g[c].transform(lambda s: s.diff(5))

---

## 8. Sensor Deprivation Scenarios (Industrial Deployment Tiers)

A production-grade Machine Learning model must be evaluated against the realities of hardware deployment. In the EDA, we established that internal thermal sensors are unobservable in standard vehicles. Furthermore, external sensors (like ambient or coolant temperature probes) are subject to failure or cost-reduction removals.

To rigorously quantify the value of our Soft-Sensor, we generate three distinct feature matrices, simulating varying degrees of sensor availability:

1. **`XA` (The Oracle / Upper-Bound Reference):** Contains full sensor availability, including the latent thermal states (`pm`, `stator_tooth`, `stator_yoke`) and their rolling histories. This model "cheats" by industrial standards, but serves as the theoretical upper bound of predictability for our system.
2. **`XB_soft` (The Standard Soft-Sensor):** The primary industrial target. It strictly excludes the internal thermal sensors and all their derived rolling features. The model must infer the thermal state purely from electromechanical telemetry and fluid boundary conditions.
3. **`XB_strict` (The Degraded Soft-Sensor):** A severe constraint scenario. It further excludes `coolant` and `ambient` temperatures. The model operates entirely blind to its thermodynamic environment, relying solely on torque, speed, and inverter electrical feedback.



By applying a strict exclusion filter, we ensure zero mathematical contamination from the forbidden sensors into the `XB` datasets.

In [111]:
forbidden_soft = ["pm", "stator_tooth", "stator_yoke"]
forbidden_strict = forbidden_soft + ["coolant", "ambient"]

feature_cols = [c for c in df.columns if c not in [TARGET, GROUP]]

def keep_col(col: str, forbidden: list[str]) -> bool:
    # drop exact sensor names and any derived feature starting with "<sensor>_"
    return all(not (col == f or col.startswith(f + "_")) for f in forbidden)

XA = df[feature_cols]
XB_soft = df[[c for c in feature_cols if keep_col(c, forbidden_soft)]]
XB_strict = df[[c for c in feature_cols if keep_col(c, forbidden_strict)]]

---

## 9. Transient Initialization & Maximizing Data Retention

The construction of rolling thermal memory inherently introduces a mathematical constraint: a moving average of size $W$ cannot be computed for the first $W-1$ timestamps of a new sequence. In signal processing, this is analogous to a filter's **"Burn-In" or transient initialization period**, resulting in `NaN` values at the start of every single `profile_id`.

> **Optimization: Per-Scenario Masking**
> The naive approach is a global drop of missing values (`df.dropna()`), which would penalize all datasets based on the largest feature window (110 seconds). However, degraded feature sets (like `XB_strict`) exclude certain long-window variables (e.g., `coolant_rm110`). 
> 
> To prevent unnecessary data attrition, we dynamically compute validity masks **independently for each feature tier** (`XA`, `XB_soft`, `XB_strict`). This adaptive masking guarantees zero missing values in the final matrices while strictly maximizing the number of available training samples for the heavily constrained models.

In [ ]:
def valid_mask(X):
    cols = [GROUP, TARGET] + list(X.columns)
    return df[cols].notna().all(axis=1)

mask_XA = valid_mask(XA)
mask_XB_soft = valid_mask(XB_soft)
mask_XB_strict = valid_mask(XB_strict)

XA = XA.loc[mask_XA].reset_index(drop=True)
XB_soft = XB_soft.loc[mask_XB_soft].reset_index(drop=True)
XB_strict = XB_strict.loc[mask_XB_strict].reset_index(drop=True)

---

## 10. Data Serialization & Artifact Generation

The final step in this pipeline is to export the engineered feature matrices into a robust, high-performance storage format for downstream modeling. We serialize the data using the **Apache Parquet** format.

**Why Parquet?**
Unlike standard CSV, Parquet is a columnar storage format that natively preserves complex data types (such as our categorical `profile_id`), significantly accelerates I/O operations, and reduces disk footprint via aggressive compression (we utilize the `zstd` algorithm).

Each exported artifact is a self-contained dataset encompassing:
1. The grouping identifier (`profile_id`), ensuring validation splits remain consistent.
2. The target variable (`stator_winding`).
3. The specific engineered feature set (`XA_full`, `XB_soft`, or `XB_strict`).

This bundled approach prevents alignment errors and guarantees absolute reproducibility during the training phase.

In [114]:
PARQ = Path("../data/parquet")
PARQ.mkdir(parents=True, exist_ok=True)

def export_dataset(X, mask, name):
    out = df.loc[mask, [GROUP, TARGET]].reset_index(drop=True)
    out = pd.concat([out, X.reset_index(drop=True)], axis=1)
    out.to_parquet(PARQ / f"{name}.parquet", engine="pyarrow", compression="zstd")

df[GROUP] = df[GROUP].astype("category")

export_dataset(XA, mask_XA, "XA_full")
export_dataset(XB_soft, mask_XB_soft, "XB_soft")
export_dataset(XB_strict, mask_XB_strict, "XB_strict")

print("Parquet export done.")
print("XA_full:", pd.read_parquet(PARQ/"XA_full.parquet").shape)
print("XB_soft:", pd.read_parquet(PARQ/"XB_soft.parquet").shape)
print("XB_strict:", pd.read_parquet(PARQ/"XB_strict.parquet").shape)

Parquet export done.
XA_full: (1315636, 105)
XB_soft: (1315636, 78)
XB_strict: (1315636, 60)


---

## 11. Final Integrity Audit (Pre-Flight Checks)

Before proceeding to the Machine Learning stages (Notebook 04), we perform a final, lightweight integrity audit on the serialized Parquet files. This ensures no hidden mathematical anomalies (`NaN` or `inf` values) were introduced during the export process.

We also output the summary statistics of the primary `XB_soft` dataset to confirm that the physical distributions (e.g., the speed-torque operating envelope) remain uncorrupted.

In [117]:
for name in ["XA_full", "XB_soft", "XB_strict"]:
    df_check = pd.read_parquet(PARQ / f"{name}.parquet")
    print(f"\n{name}")
    print("Shape:", df_check.shape)
    print("NaN count:", df_check.isna().sum().sum())
    print("Inf count:", np.isinf(df_check.select_dtypes(np.number)).sum().sum())

df_tmp = pd.read_parquet(PARQ / "XB_soft.parquet")

display(df_tmp[[TARGET, "motor_speed", "torque"]].describe(percentiles=[.01,.5,.99]).T)


XA_full
Shape: (1315636, 105)
NaN count: 0
Inf count: 0

XB_soft
Shape: (1315636, 78)
NaN count: 0
Inf count: 0

XB_strict
Shape: (1315636, 60)
NaN count: 0
Inf count: 0


,count,mean,std,min,1%,50%,99%,max
stator_winding,1315636.0,66.652288,28.615282,19.179312,19.549914,65.435346,124.416554,141.362885
motor_speed,1315636.0,2198.409668,1858.166448,-275.549144,-0.005001,1999.976685,5999.934082,6000.015137
torque,1315636.0,30.858926,76.921119,-246.466663,-178.466380,10.841741,213.196686,260.624603


## 12. Executive Summary: From Telemetry to Thermodynamic State-Space

In this notebook, we successfully bridged the gap between raw, instantaneous sensor telemetry and the physical realities of thermal dynamics. We engineered a robust, tabular dataset that natively embeds the motor's thermal memory without compromising operational causality.

**Key Achievements:**
1. **Physical State-Space Reconstruction:** We derived instantaneous proxies for active electrical power and mechanical load, implicitly allowing future models to learn the motor's power loss equations.
2. **Causal Thermal Memory:** By utilizing strictly shifted (`shift(1)`), profile-isolated rolling statistics, we captured the system's thermal inertia ($\tau \approx 110\text{s}$) while mathematically guaranteeing zero future-to-past data leakage.
3. **Deployment-Constrained Scenarios:** We formulated three distinct feature matrices (`XA_full` [Oracle], `XB_soft` [Standard], `XB_strict` [Degraded]) serialized in optimized Parquet format, preparing our pipeline for rigorous, industrial-grade validation.

---

## 13. Next Steps: Operating Regime Discovery

With our physics-informed feature matrices fully constructed and serialized, the data is structurally ready for supervised predictive modeling. 

However, predicting temperature is fundamentally different when the motor is in a high-torque acceleration phase (dominated by Copper Losses) versus a high-speed cruising phase (dominated by Iron Losses). 

**Next Notebook (`03_unsupervised_regime_discovery.ipynb`):** Before training predictive algorithms, we will perform an Unsupervised Regime Discovery on the electromechanical phase space. This will allow us to automatically segment the motor's operation into discrete physical regimes, providing a vital framework for stratified error analysis in our downstream supervised models.